# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

- **Title**: A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa
- **URL**: https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json
- **Description**: The dataset contains survey data on mental health indicators among residents of Kilifi County. It includes demographic details and measurements of psychological symptoms along with scores from assessments such as GAD-7, PHQ-9, and PSQ.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata # Note: access as an object, not subscript

print(f"{metadata.name}: {metadata.description}")
print("\u2022 Published:", getattr(metadata, 'datePublished', 'N/A'))
print("\u2022 License:", getattr(metadata, 'license', 'N/A'))
print("\u2022 Keywords:", getattr(metadata, 'keywords', 'N/A'))

## 2. Data Overview
Review available record sets, fields, columns and their IDs.

We'll display information about record sets, including their `@id`, and show example records.

In [ ]:
# List available record sets in the dataset
record_sets = []

if hasattr(metadata, 'recordSet') and metadata.recordSet:
    record_sets = list(metadata.recordSet)
    print("Available Record Sets (@id):")
    for rs in record_sets:
        print("-", rs['@id'] if '@id' in rs else str(rs))
else:
    print("No record sets found in metadata. Inspecting underlying schema...")
    # Try to discover record sets directly from schema
    # If mlcroissant exposes the schema, check its keys
    try:
        schema_json = dataset.metadata.to_json()
        if 'recordSet' in schema_json and schema_json['recordSet']:
            record_sets = schema_json['recordSet']
            for rs in record_sets:
                if isinstance(rs, dict):
                    print("-", rs.get('@id', str(rs)))
                else:
                    print("-", str(rs))
        else:
            print("No recordSet found in schema JSON.")
    except Exception as e:
        print("Could not extract record sets. Error:", e)

# Try previewing records from a record set
example_record_set_id = None
if record_sets:
    # Use the first record set for demonstration
    if isinstance(record_sets[0], dict):
        example_record_set_id = record_sets[0]['@id']
    else:
        example_record_set_id = record_sets[0]
    print(f"\nPreviewing first 2 records from Record Set {example_record_set_id}:")
    try:
        for i, rec in enumerate(dataset.records(record_set=example_record_set_id)):
            print(f"Record {i+1}:")
            pprint.pprint(rec)
            if i == 1: break
    except Exception as e:
        print(f"Failed to load records for {example_record_set_id}:", e)

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Compile list of record set @id for extraction
record_set_ids = []
# Use detected record set ids from previous step
for rs in record_sets:
    if isinstance(rs, dict):
        record_set_ids.append(rs['@id'])
    else:
        record_set_ids.append(rs)

# Extract records into pandas DataFrames
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nColumns in DataFrame for Record Set {record_set_id}:\n", df.columns.tolist())
        print("Sample records:")
        print(df.head(2))
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# For demonstration, pick the primary record set
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    df_main = dataframes[main_record_set_id]
else:
    df_main = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll:
- Filter respondents with high PHQ-9 scores (indicative of depressive symptoms).
- Normalize the PHQ-9 scores.
- Group results by `level_of_education`.

In [ ]:
# EDA: Analyze PHQ-9 Scores
# Record set and field @ids should be used
phq9_field_id = 'phq_9_score'        # Replace with actual @id from schema if available
education_field_id = 'level_of_education' # Replace with actual @id from schema if available

if df_main is not None:
    # If columns are @id, ensure we reference by that
    columns = df_main.columns.tolist()

    # Try to identify exact column names (@id preferred)
    # Common heuristics: 'phq_9_score', 'PHQ9', etc.
    def find_col_by_keywords(df_cols, keywords):
        for c in df_cols:
            for kw in keywords:
                if kw.lower() in c.lower():
                    return c
        return None

    phq9_col = find_col_by_keywords(columns, ['phq', 'phq9', 'phq_9']) or phq9_field_id
    edu_col = find_col_by_keywords(columns, ['education', 'level_of_education']) or education_field_id

    # Remove missing values for these fields
    filtered_df = df_main[df_main[phq9_col].notnull()]
    try:
        filtered_df[phq9_col] = pd.to_numeric(filtered_df[phq9_col], errors='coerce')
    except Exception:
        pass

    threshold = 10
    high_phq9_df = filtered_df[filtered_df[phq9_col] > threshold]
    print(f"Filtered records with {phq9_col} > {threshold} (possible moderate/severe depression):")
    print(high_phq9_df[[phq9_col, edu_col]].head())

    # Normalize PHQ-9 scores
    if not high_phq9_df.empty:
        high_phq9_df[f"{phq9_col}_normalized"] = (high_phq9_df[phq9_col] - high_phq9_df[phq9_col].mean()) / high_phq9_df[phq9_col].std()
        print(f"\nNormalized {phq9_col} for filtered records:")
        print(high_phq9_df[[phq9_col, f"{phq9_col}_normalized"]].head())

        # Group by education field if present
        if edu_col in high_phq9_df.columns:
            grouped_df = high_phq9_df.groupby(edu_col)[phq9_col].mean().reset_index().sort_values(phq9_col, ascending=False)
            print(f"\nAverage {phq9_col} by {edu_col}:")
            print(grouped_df.head())
else:
    print("No main DataFrame loaded for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll show:
- Histogram of PHQ-9 scores
- Boxplot of PHQ-9 scores by level of education

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df_main is not None and phq9_col in df_main.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df_main[phq9_col].dropna(), bins=15, kde=True)
    plt.title("Distribution of PHQ-9 Scores")
    plt.xlabel("PHQ-9 Score")
    plt.ylabel("Count")
    plt.show()

    if edu_col in df_main.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=df_main[edu_col], y=df_main[phq9_col])
        plt.title(f"PHQ-9 Scores by {edu_col}")
        plt.xlabel(edu_col)
        plt.ylabel("PHQ-9 Score")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides valuable insights into mental health among residents of Kilifi County, Kenya.
- PHQ-9 scores indicate a proportion of the population may experience moderate to severe depressive symptoms; levels vary across education groups.
- The `mlcroissant` library enables structured and reproducible access to data by referencing entities using their `@id` fields.

Further analyses could include correlations between demographic and assessment scores, and insights to inform local health policy and interventions.